# Savegame to Pandas

Load an EU5 savegame and extract data into pandas DataFrames for analysis.

In [ ]:
import pandas as pd

from analysis.savegame import load_save, get_locations_df, get_countries_df, get_buildings_df
from analysis.building_levels.building_analysis.utils import load_config
from analysis.savegame.loader import get_latest_save_path
from core.parser.path_resolver import PathResolver
from core.data.location_data import LocationData

config = load_config()
save_dir = config.get("save_games_dir")
latest_path = get_latest_save_path(save_dir) if save_dir else None
print(f"Latest save: {latest_path}")

In [ ]:
# Load save (uses latest from config if path=None; pass path= for a specific file)
save = load_save(path=latest_path)
# print(f"Loaded: {save.name}, date: {save.game_date}")


In [ ]:
save._data["locations"]["locations"]["1"]

In [ ]:
# Load location hierarchy for scope merge (super_region, macro_region, region, area, province)
path_resolver = PathResolver(config["game_path"], config["mod_path"])
location_data = LocationData(path_resolver)
location_data.load_all()

In [ ]:
# Build locations DataFrame and attach as save._datalocations (merge with scope hierarchy)
df_locs = get_locations_df(save)
df_scopes = location_data.get_merged_df()[["location", "super_region", "macro_region", "region", "area", "province"]].drop_duplicates(subset="location")
save._datalocations = df_locs.merge(df_scopes, left_on="slug", right_on="location", how="left").drop(columns=["location"])
scope_cols = ["province", "area", "region", "macro_region", "super_region"]
other_cols = [c for c in save._datalocations.columns if c not in ["slug"] + scope_cols]
save._datalocations = save._datalocations[["slug"] + [c for c in scope_cols if c in save._datalocations.columns] + other_cols]

In [ ]:
save._datalocations.loc[save._datalocations.slug == "stockholm"].iloc[:, 11:25]

In [ ]:
save._datalocations.loc[save._datalocations.slug == "stockholm"][["total_population"]]

In [ ]:
# save._datalocations.to_pickle(path="save_game_temp/game_start.pkl")
# save._datalocations.to_pickle(path="save_game_temp/pre_black_death_1346.pkl")
# save._datalocations.to_pickle(path="save_game_temp/post_black_death_1354_90m.pkl")
save._datalocations.to_pickle(path="save_game_temp/1400.pkl")

## Locations

In [ ]:
# save._datalocations.iloc[:, :29]

In [ ]:
# save._datalocations.groupby("religion", dropna=False)[["development", "total_population"]].sum().sort_values("total_population", ascending = False).round(0).head(15)
save._datalocations.groupby("super_region", dropna=False)[["development", "total_population"]].sum().sort_values("total_population", ascending = False).round(0).head(15)

## Countries

In [ ]:
df_countries = get_countries_df(save)
df_countries.head(10)
# save._data["countries"]["database"]["3"] # swe